# ACI 1.3.2 publication audit

Reproduce catalog, source-version and score checks from the local accepted publication. The notebook never fetches data or changes the index. AA display registries are intentionally outside the public fit and bulk files.

In [1]:
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/index-config.yaml").exists())
snapshots = sorted((root / "data/snapshots").glob("????-??-??/snapshot.json"), reverse=True)
snapshot = json.loads(snapshots[0].read_text())
runs = [r for r in snapshot["runs"] if r["methodVersion"].endswith("1.3.2")]
assert {r["kind"] for r in runs} == {"mixed", "agentic", "chat"}
latest = {kind: max((r for r in runs if r["kind"] == kind), key=lambda r: r["createdAt"]) for kind in ("mixed", "agentic", "chat")}
print("Catalog models:", len(snapshot["models"]))
for kind, run in latest.items():
    rows = [s for s in snapshot["scores"] if s["runId"] == run["id"]]
    print(kind, "estimated models:", len({s["modelId"] for s in rows}), "systems:", len(rows))


Catalog models: 126
mixed estimated models: 108 systems: 119
agentic estimated models: 108 systems: 119
chat estimated models: 108 systems: 119


## Same-method GPT-5.5 comparison
A median difference alone does not establish superiority. The source audit corrects FrontierMath revisions and Pro effort settings. At matched xhigh, Pro leads both FrontierMath subsets; ARC-AGI-2 slightly favors base.

In [2]:
for kind, run in latest.items():
    rows = [s for s in snapshot["scores"] if s["runId"] == run["id"] and s["modelId"] in ("gpt-5.5", "gpt-5.5-pro")]
    print(kind)
    for row in rows:
        print(row["modelId"], row.get("profile"), (row.get("score") if row.get("score") is not None else row.get("robustScore")), row["ciLow"], row["ciHigh"], row.get("tier"))


mixed
gpt-5.5 std-common 57.34143 51.891804 62.584274 provisional
gpt-5.5 max-common 64.924904 57.039654 74.82623 ranked
gpt-5.5-pro std-common 60.48693 50.596504 70.1403 ranked
gpt-5.5-pro max-common 68.33346 56.0425 81.79485 provisional
agentic
gpt-5.5 std-common 56.80576 50.493904 62.417965 provisional
gpt-5.5 max-common 61.17766 55.02755 68.48995 ranked
gpt-5.5-pro std-common 61.41367 51.601807 71.17535 ranked
gpt-5.5-pro max-common 66.45868 55.110287 78.67117 provisional
chat
gpt-5.5 std-common 56.163044 48.238434 63.14272 provisional
gpt-5.5 max-common 65.52909 52.904457 81.42792 ranked
gpt-5.5-pro std-common 58.350567 44.880127 71.14021 ranked
gpt-5.5-pro max-common 67.87151 50.374695 87.13915 provisional


## Version and publication boundary checks
The new evidence inventory prevents rows from earlier incorrect imports reappearing in current tables. Historical runs and their score metadata are retained.

In [3]:
benchmarks = {b["id"]: b for b in snapshot["benchmarks"]}
assert "matharena-composite" in benchmarks
assert not {"swe-bench-pro-public", "livecodebench-v6-pro", "gsm8k", "aime-2025"} & set(benchmarks)
assert benchmarks["frontiermath-v2-tiers-1-3"]["nItems"] == 285
assert benchmarks["frontiermath-v2-tier-4"]["nItems"] == 41
for result in snapshot["results"]:
    if result["benchmarkId"].startswith("frontiermath-v2"):
        if result["sourceId"] == "epoch":
            assert result["graderVersion"] == "2.0.0", result
            assert result["config"]["dataset"] == "private", result
        else:
            assert str(result["observedOn"]) >= "2026-06-12", result
    assert result["sourceId"] != "aa-benchmarks-manual"
for run in latest.values():
    assert isinstance(run["params"]["current_evidence_observation_keys"], list)
assert "speed" not in snapshot
print("Version and display-data boundaries passed.")


Version and display-data boundaries passed.
